In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_new = pd.read_csv('WELFake_Dataset.csv')

print("Shape:", df_new.shape)
print("Columns:", df_new.columns.tolist())
print("Label distribution:")
print(df_new['label'].value_counts())
df_new.head()

Shape: (72134, 4)
Columns: ['Unnamed: 0', 'title', 'text', 'label']
Label distribution:
label
1    37106
0    35028
Name: count, dtype: int64


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [15]:
import re
import string

# Drop unnecessary column
df_new = df_new.drop(columns=['Unnamed: 0'])

# Drop rows with missing values
df_new = df_new.dropna()

print("After cleaning shape:", df_new.shape)
print("Missing values:", df_new.isnull().sum())

# Combine title + text
df_new['content'] = df_new['title'] + ' ' + df_new['text']

# Clean text function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\n', '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    return text

# Apply cleaning
df_new['content'] = df_new['content'].apply(clean_text)

print("\nCleaning done!")
print(df_new['content'].head())

After cleaning shape: (71537, 3)
Missing values: title    0
text     0
label    0
dtype: int64

Cleaning done!
0    law enforcement on high alert following threat...
2    unbelievable obama’s attorney general says mos...
3    bobby jindal raised hindu uses story of christ...
4    satan  russia unvelis an image of its terrifyi...
5    about time christian group sues amazon and spl...
Name: content, dtype: str


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Split data
X = df_new['content']
y = df_new['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # ensures balanced split
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("\nTrain label distribution:")
print(y_train.value_counts())

Train size: (57229,)
Test size: (14308,)

Train label distribution:
label
1    29207
0    28022
Name: count, dtype: int64


In [17]:
# TF-IDF Vectorization
tfidf_new = TfidfVectorizer(
    max_features=10000, 
    stop_words='english',
    ngram_range=(1,2)  # captures word pairs too
)

X_train_tfidf = tfidf_new.fit_transform(X_train)
X_test_tfidf = tfidf_new.transform(X_test)

print("TF-IDF done!")
print("Shape:", X_train_tfidf.shape)

# Train Logistic Regression
model_new = LogisticRegression(max_iter=1000)
model_new.fit(X_train_tfidf, y_train)

print("Model trained!")

# Check Accuracy
y_pred = model_new.predict(X_test_tfidf)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Fake', 'Real']))

TF-IDF done!
Shape: (57229, 10000)
Model trained!

Accuracy: 0.9481409001956947

Classification Report:
              precision    recall  f1-score   support

        Fake       0.96      0.93      0.95      7006
        Real       0.94      0.96      0.95      7302

    accuracy                           0.95     14308
   macro avg       0.95      0.95      0.95     14308
weighted avg       0.95      0.95      0.95     14308



In [18]:
# Test on custom news
test_news = [
    "NASA astronauts successfully returned to Earth after completing a six-month mission aboard the International Space Station.",
    "Scientists discover that drinking coffee makes you immortal. Government has been hiding this secret for decades.",
    "President Biden signed a new executive order directing federal agencies to reduce government spending.",
    "Doctors don't want you to know this one weird trick that cures all diseases instantly."
]

for news in test_news:
    cleaned = clean_text(news)
    vec = tfidf_new.transform([cleaned])
    pred = model_new.predict(vec)[0]
    conf = model_new.predict_proba(vec)[0]
    label = " REAL" if pred == 1 else "🚨 FAKE"
    print(f"{label} ({conf[pred]*100:.2f}%) → {news[:60]}...")
    print()

 REAL (80.11%) → NASA astronauts successfully returned to Earth after complet...

 REAL (90.76%) → Scientists discover that drinking coffee makes you immortal....

 REAL (63.50%) → President Biden signed a new executive order directing feder...

 REAL (83.18%) → Doctors don't want you to know this one weird trick that cur...



In [19]:
# Check prediction distribution on test set
print("Prediction distribution:")
print(pd.Series(y_pred).value_counts())

# Check probability threshold
for news in test_news:
    cleaned = clean_text(news)
    vec = tfidf_new.transform([cleaned])
    conf = model_new.predict_proba(vec)[0]
    print(f"Fake: {conf[0]*100:.2f}% | Real: {conf[1]*100:.2f}% → {news[:50]}...")

Prediction distribution:
1    7488
0    6820
Name: count, dtype: int64
Fake: 19.89% | Real: 80.11% → NASA astronauts successfully returned to Earth aft...
Fake: 9.24% | Real: 90.76% → Scientists discover that drinking coffee makes you...
Fake: 36.50% | Real: 63.50% → President Biden signed a new executive order direc...
Fake: 16.82% | Real: 83.18% → Doctors don't want you to know this one weird tric...


In [20]:
# Smart prediction function
def predict_news(text, threshold_fake=0.45, threshold_real=0.60):
    cleaned = clean_text(text)
    vec = tfidf_new.transform([cleaned])
    conf = model_new.predict_proba(vec)[0]
    
    fake_conf = conf[0] * 100
    real_conf = conf[1] * 100
    
    if conf[0] >= threshold_fake:
        verdict = "🚨 FAKE"
    elif conf[1] >= threshold_real:
        verdict = "✅ REAL"
    else:
        verdict = "⚠️ UNVERIFIED"
    
    return verdict, fake_conf, real_conf

# Test all news
test_news = [
    "NASA astronauts successfully returned to Earth after completing a six-month mission aboard the International Space Station.",
    "Scientists discover that drinking coffee makes you immortal. Government has been hiding this secret for decades.",
    "President Biden signed a new executive order directing federal agencies to reduce government spending.",
    "Doctors don't want you to know this one weird trick that cures all diseases instantly.",
    "The Federal Reserve raised interest rates by 0.25% following inflation concerns reported by Reuters.",
    "SHOCKING: Bill Gates microchips found in COVID vaccines, whistleblower reveals secret agenda."
]

print("=" * 60)
for news in test_news:
    verdict, fake_conf, real_conf = predict_news(news)
    print(f"{verdict}")
    print(f"Fake: {fake_conf:.2f}% | Real: {real_conf:.2f}%")
    print(f"News: {news[:60]}...")
    print("-" * 60)

✅ REAL
Fake: 19.89% | Real: 80.11%
News: NASA astronauts successfully returned to Earth after complet...
------------------------------------------------------------
✅ REAL
Fake: 9.24% | Real: 90.76%
News: Scientists discover that drinking coffee makes you immortal....
------------------------------------------------------------
✅ REAL
Fake: 36.50% | Real: 63.50%
News: President Biden signed a new executive order directing feder...
------------------------------------------------------------
✅ REAL
Fake: 16.82% | Real: 83.18%
News: Doctors don't want you to know this one weird trick that cur...
------------------------------------------------------------
🚨 FAKE
Fake: 95.51% | Real: 4.49%
News: The Federal Reserve raised interest rates by 0.25% following...
------------------------------------------------------------
✅ REAL
Fake: 1.44% | Real: 98.56%
News: SHOCKING: Bill Gates microchips found in COVID vaccines, whi...
------------------------------------------------------------


In [21]:
# Check what Real news looks like in WELFake
print("=== REAL NEWS SAMPLES (label=1) ===")
for i in range(3):
    print(df_new[df_new['label']==1]['title'].iloc[i])
    print()

print("=== FAKE NEWS SAMPLES (label=0) ===")
for i in range(3):
    print(df_new[df_new['label']==0]['title'].iloc[i])
    print()

=== REAL NEWS SAMPLES (label=1) ===
LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO]

UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MOST CHARLOTTE RIOTERS WERE “PEACEFUL” PROTESTERS…In Her Home State Of North Carolina [VIDEO]

SATAN 2: Russia unvelis an image of its terrifying new ‘SUPERNUKE’ – Western world takes notice

=== FAKE NEWS SAMPLES (label=0) ===
Bobby Jindal, raised Hindu, uses story of Christian conversion to woo evangelicals for potential 2016 bid

May Brexit offer would hurt, cost EU citizens - EU parliament

Schumer calls on Trump to appoint official to oversee Puerto Rico relief



In [22]:
# Flip labels
df_new['label'] = df_new['label'].map({1: 0, 0: 1})

# Verify
print("After flipping:")
print(df_new['label'].value_counts())

# Check samples
print("\n=== Label 1 (should be REAL now) ===")
for i in range(2):
    print(df_new[df_new['label']==1]['title'].iloc[i])
    print()

print("=== Label 0 (should be FAKE now) ===")
for i in range(2):
    print(df_new[df_new['label']==0]['title'].iloc[i])
    print()

After flipping:
label
0    36509
1    35028
Name: count, dtype: int64

=== Label 1 (should be REAL now) ===
Bobby Jindal, raised Hindu, uses story of Christian conversion to woo evangelicals for potential 2016 bid

May Brexit offer would hurt, cost EU citizens - EU parliament

=== Label 0 (should be FAKE now) ===
LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO]

UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MOST CHARLOTTE RIOTERS WERE “PEACEFUL” PROTESTERS…In Her Home State Of North Carolina [VIDEO]



In [23]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Split data
X = df_new['content']
y = df_new['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# TF-IDF
tfidf_new = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    ngram_range=(1,2)
)

X_train_tfidf = tfidf_new.fit_transform(X_train)
X_test_tfidf = tfidf_new.transform(X_test)

# Train model
model_new = LogisticRegression(max_iter=1000)
model_new.fit(X_train_tfidf, y_train)

# Accuracy
y_pred = model_new.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Fake', 'Real']))

Accuracy: 0.9464635169136147

Classification Report:
              precision    recall  f1-score   support

        Fake       0.94      0.95      0.95      7302
        Real       0.95      0.94      0.94      7006

    accuracy                           0.95     14308
   macro avg       0.95      0.95      0.95     14308
weighted avg       0.95      0.95      0.95     14308



In [24]:
# Smart prediction function
def predict_news(text):
    cleaned = clean_text(text)
    vec = tfidf_new.transform([cleaned])
    conf = model_new.predict_proba(vec)[0]
    
    fake_conf = conf[0] * 100
    real_conf = conf[1] * 100
    
    if fake_conf >= 55:
        verdict = "🚨 FAKE"
    elif real_conf >= 55:
        verdict = "✅ REAL"
    else:
        verdict = "⚠️ UNVERIFIED"
    
    return verdict, fake_conf, real_conf

# Test all news
test_news = [
    "NASA astronauts successfully returned to Earth after completing a six-month mission aboard the International Space Station.",
    "Scientists discover that drinking coffee makes you immortal. Government has been hiding this secret for decades.",
    "President Biden signed a new executive order directing federal agencies to reduce government spending.",
    "SHOCKING: Bill Gates microchips found in COVID vaccines, whistleblower reveals secret agenda.",
    "The Federal Reserve raised interest rates by 0.25% following inflation concerns reported by Reuters.",
    "UNBELIEVABLE! Government hiding secret cure for cancer from public for decades!"
]

print("=" * 65)
for news in test_news:
    verdict, fake_conf, real_conf = predict_news(news)
    print(f"{verdict}")
    print(f"Fake: {fake_conf:.2f}% | Real: {real_conf:.2f}%")
    print(f"News: {news[:60]}...")
    print("-" * 65)

🚨 FAKE
Fake: 81.49% | Real: 18.51%
News: NASA astronauts successfully returned to Earth after complet...
-----------------------------------------------------------------
🚨 FAKE
Fake: 87.97% | Real: 12.03%
News: Scientists discover that drinking coffee makes you immortal....
-----------------------------------------------------------------
⚠️ UNVERIFIED
Fake: 52.11% | Real: 47.89%
News: President Biden signed a new executive order directing feder...
-----------------------------------------------------------------
🚨 FAKE
Fake: 98.39% | Real: 1.61%
News: SHOCKING: Bill Gates microchips found in COVID vaccines, whi...
-----------------------------------------------------------------
✅ REAL
Fake: 4.66% | Real: 95.34%
News: The Federal Reserve raised interest rates by 0.25% following...
-----------------------------------------------------------------
🚨 FAKE
Fake: 94.04% | Real: 5.96%
News: UNBELIEVABLE! Government hiding secret cure for cancer from ...
------------------------------------

In [25]:
def predict_news(text):
    cleaned = clean_text(text)
    vec = tfidf_new.transform([cleaned])
    conf = model_new.predict_proba(vec)[0]
    
    fake_conf = conf[0] * 100
    real_conf = conf[1] * 100

    if fake_conf >= 65:        # raised from 55 to 65
        verdict = "🚨 FAKE"
    elif real_conf >= 45:      # lowered from 55 to 45
        verdict = "✅ REAL"
    else:
        verdict = "⚠️ UNVERIFIED"
    
    return verdict, fake_conf, real_conf

# Test again
test_news = [
    "NASA astronauts successfully returned to Earth after completing a six-month mission aboard the International Space Station.",
    "Scientists discover that drinking coffee makes you immortal. Government has been hiding this secret for decades.",
    "President Biden signed a new executive order directing federal agencies to reduce government spending.",
    "SHOCKING: Bill Gates microchips found in COVID vaccines, whistleblower reveals secret agenda.",
    "The Federal Reserve raised interest rates by 0.25% following inflation concerns reported by Reuters.",
    "UNBELIEVABLE! Government hiding secret cure for cancer from public for decades!"
]

print("=" * 65)
for news in test_news:
    verdict, fake_conf, real_conf = predict_news(news)
    print(f"{verdict}")
    print(f"Fake: {fake_conf:.2f}% | Real: {real_conf:.2f}%")
    print(f"News: {news[:60]}...")
    print("-" * 65)

🚨 FAKE
Fake: 81.49% | Real: 18.51%
News: NASA astronauts successfully returned to Earth after complet...
-----------------------------------------------------------------
🚨 FAKE
Fake: 87.97% | Real: 12.03%
News: Scientists discover that drinking coffee makes you immortal....
-----------------------------------------------------------------
✅ REAL
Fake: 52.11% | Real: 47.89%
News: President Biden signed a new executive order directing feder...
-----------------------------------------------------------------
🚨 FAKE
Fake: 98.39% | Real: 1.61%
News: SHOCKING: Bill Gates microchips found in COVID vaccines, whi...
-----------------------------------------------------------------
✅ REAL
Fake: 4.66% | Real: 95.34%
News: The Federal Reserve raised interest rates by 0.25% following...
-----------------------------------------------------------------
🚨 FAKE
Fake: 94.04% | Real: 5.96%
News: UNBELIEVABLE! Government hiding secret cure for cancer from ...
-------------------------------------------

In [26]:
import pickle

# Save new model and vectorizer
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_new, f)

with open('logistic_model.pkl', 'wb') as f:
    pickle.dump(model_new, f)

print("Models saved successfully! ✅")

Models saved successfully! ✅
